In [1]:
# Cell 1 — Imports
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from collections import defaultdict
import random
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")

All imports successful


In [2]:
# Cell 2 — Build the Deck & Core Game Logic (Python translation of Java project)
# Mirrors the BlackJackToFive Java implementation from COMP 2000

SUITS = ['♠', '♥', '♦', '♣']
RANKS = ['2','3','4','5','6','7','8','9','10','J','Q','K','A']
RANK_VALUES = {
    '2':2,'3':3,'4':4,'5':5,'6':6,'7':7,'8':8,'9':9,'10':10,
    'J':10,'Q':10,'K':10,'A':11
}

def build_deck():
    return [(r, s) for s in SUITS for r in RANKS]

def hand_value(hand):
    """Mirrors BlackJackUtils ace handling — hard ace to soft ace if bust."""
    value = sum(RANK_VALUES[r] for r, s in hand)
    aces  = sum(1 for r, s in hand if r == 'A')
    while value > 21 and aces:
        value -= 10
        aces  -= 1
    return value

def is_blackjack(hand):
    return hand_value(hand) == 21 and len(hand) == 2

# Test
deck = build_deck()
random.shuffle(deck)
test_hand = [deck.pop(), deck.pop()]
print(f"Deck size:      {len(deck) + 2} cards")
print(f"Test hand:      {test_hand}")
print(f"Hand value:     {hand_value(test_hand)}")
print(f"Is blackjack:   {is_blackjack(test_hand)}")
print(f"\nCore game logic verified ✓")

Deck size:      52 cards
Test hand:      [('8', '♠'), ('10', '♠')]
Hand value:     18
Is blackjack:   False

Core game logic verified ✓


In [3]:
# Cell 3 — Simulate 10,000 BlackJack Games (Monte Carlo)
def simulate_game(n_players=3, target_wins=5, strategy='basic'):
    """
    Simulate one full BlackJack-to-Five game.
    strategy: 'basic' = hit under 17, 'aggressive' = hit under 19
    """
    deck = build_deck()
    random.shuffle(deck)
    
    wins = defaultdict(int)
    dealer_idx = 0
    rounds = 0
    hit_threshold = 17 if strategy == 'basic' else 19
    
    while max(wins.values(), default=0) < target_wins:
        if len(deck) < n_players * 4:
            deck = build_deck()
            random.shuffle(deck)
        
        hands = {i: [deck.pop(), deck.pop()] for i in range(n_players)}
        rounds += 1
        
        # Players hit or stand
        for i in range(n_players):
            if i == dealer_idx:
                continue
            while hand_value(hands[i]) < hit_threshold and len(deck) > 0:
                hands[i].append(deck.pop())
        
        # Dealer plays
        while hand_value(hands[dealer_idx]) < 17 and len(deck) > 0:
            hands[dealer_idx].append(deck.pop())
        
        dealer_val = hand_value(hands[dealer_idx])
        
        # Determine round winner
        best_val = 0
        winner   = dealer_idx
        for i in range(n_players):
            v = hand_value(hands[i])
            if v <= 21 and v > best_val:
                best_val = v
                winner   = i
        
        # Dealer wins ties (house advantage)
        if best_val == dealer_val:
            winner = dealer_idx
            
        wins[winner] += 1
        dealer_idx = (dealer_idx + 1) % n_players
    
    return wins, rounds

# Run 10,000 simulations
N_SIMS    = 10_000
N_PLAYERS = 3
results   = []

print(f"Running {N_SIMS:,} simulations with {N_PLAYERS} players...")
for _ in range(N_SIMS):
    wins, rounds = simulate_game(n_players=N_PLAYERS)
    winner = max(wins, key=wins.get)
    results.append({'winner': winner, 'rounds': rounds, 'wins': wins[winner]})

results_df = pd.DataFrame(results)

print(f"\n=== Simulation Results ===")
print(f"Avg rounds per game:     {results_df['rounds'].mean():.1f}")
print(f"Avg rounds to 5 wins:    {results_df['rounds'].mean():.1f}")
print(f"\nWin rate by player position:")
for p in range(N_PLAYERS):
    rate = (results_df['winner'] == p).mean()
    label = '(dealer start)' if p == 0 else ''
    print(f"  Player {p} {label}: {rate*100:.1f}%")

Running 10,000 simulations with 3 players...

=== Simulation Results ===
Avg rounds per game:     9.7
Avg rounds to 5 wins:    9.7

Win rate by player position:
  Player 0 (dealer start): 39.5%
  Player 1 : 32.9%
  Player 2 : 27.7%


In [4]:
# Cell 4 — Probability Analysis: House Edge & Bust Rates
def bust_probability(n_simulations=50000):
    """Calculate bust probability for different hand values."""
    bust_rates = {}
    
    for start_value in range(12, 22):
        busts = 0
        for _ in range(n_simulations):
            deck = build_deck()
            random.shuffle(deck)
            busts += 1 if RANK_VALUES.get(deck[0][0], 10) + start_value > 21 else 0
        bust_rates[start_value] = busts / n_simulations
    
    return bust_rates

print("Calculating bust probabilities...")
bust_rates = bust_probability()

bust_df = pd.DataFrame([
    {'hand_value': k, 'bust_probability': v}
    for k, v in bust_rates.items()
])

fig = px.bar(bust_df, x='hand_value', y='bust_probability',
             color='bust_probability',
             color_continuous_scale='Reds',
             title='Bust Probability When Hitting at Each Hand Value',
             labels={'hand_value': 'Current Hand Value',
                     'bust_probability': 'Probability of Bust'},
             template='plotly_dark')
fig.update_layout(width=750, height=430)
fig.show()

print("\nBust probability table:")
print(bust_df.to_string(index=False))

Calculating bust probabilities...



Bust probability table:
 hand_value  bust_probability
         12           0.38918
         13           0.45914
         14           0.54120
         15           0.61346
         16           0.69404
         17           0.76786
         18           0.84746
         19           0.92416
         20           1.00000
         21           1.00000


In [5]:
# Cell 5 — Optimal Strategy: Dynamic Programming
def optimal_strategy_dp():
    """
    Use dynamic programming to find optimal hit/stand threshold
    by simulating win rates at each strategy threshold.
    """
    strategies = range(12, 21)
    strategy_results = []
    
    for threshold in strategies:
        wins = 0
        n_games = 5000
        for _ in range(n_games):
            deck = build_deck()
            random.shuffle(deck)
            
            # Player hand
            hand = [deck.pop(), deck.pop()]
            while hand_value(hand) < threshold and len(deck) > 0:
                hand.append(deck.pop())
            player_val = hand_value(hand)
            
            # Dealer hand (always hits below 17)
            dealer_hand = [deck.pop(), deck.pop()]
            while hand_value(dealer_hand) < 17 and len(deck) > 0:
                dealer_hand.append(deck.pop())
            dealer_val = hand_value(dealer_hand)
            
            # Win condition
            if player_val <= 21:
                if dealer_val > 21 or player_val > dealer_val:
                    wins += 1
        
        win_rate = wins / n_games
        strategy_results.append({
            'threshold': threshold,
            'win_rate': round(win_rate, 4),
            'strategy': f'Hit below {threshold}'
        })
    
    return pd.DataFrame(strategy_results)

print("Running dynamic programming strategy optimization...")
strategy_df = optimal_strategy_dp()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=strategy_df['threshold'],
    y=strategy_df['win_rate'],
    mode='lines+markers',
    line=dict(color='cyan', width=2.5),
    marker=dict(size=8),
    name='Win Rate'
))

# Mark optimal threshold
best = strategy_df.loc[strategy_df['win_rate'].idxmax()]
fig.add_trace(go.Scatter(
    x=[best['threshold']],
    y=[best['win_rate']],
    mode='markers',
    marker=dict(color='lime', size=14, symbol='star'),
    name=f'Optimal: Hit below {int(best["threshold"])}'
))

fig.update_layout(
    title='Optimal Hit/Stand Strategy — Dynamic Programming',
    xaxis_title='Hit Threshold (stand at or above this value)',
    yaxis_title='Win Rate vs Dealer',
    template='plotly_dark',
    width=800, height=450
)
fig.show()

print(f"\nOptimal strategy: {best['strategy']}")
print(f"Win rate at optimal: {best['win_rate']*100:.1f}%")
print(strategy_df.to_string(index=False))

Running dynamic programming strategy optimization...



Optimal strategy: Hit below 13
Win rate at optimal: 42.6%
 threshold  win_rate     strategy
        12    0.4150 Hit below 12
        13    0.4262 Hit below 13
        14    0.4164 Hit below 14
        15    0.4214 Hit below 15
        16    0.4106 Hit below 16
        17    0.4040 Hit below 17
        18    0.3812 Hit below 18
        19    0.3546 Hit below 19
        20    0.2968 Hit below 20


In [6]:
# Cell 6 — Strategy Comparison & Export
import os

# Compare basic vs aggressive vs optimal
strategies_compare = []
for name, threshold in [('Conservative (hit<17)', 17),
                         ('Basic (hit<18)', 18),
                         ('Aggressive (hit<19)', 19),
                         ('Optimal (DP)', int(best['threshold']))]:
    wins = 0
    n = 5000
    for _ in range(n):
        deck = build_deck()
        random.shuffle(deck)
        hand = [deck.pop(), deck.pop()]
        while hand_value(hand) < threshold and len(deck) > 0:
            hand.append(deck.pop())
        player_val = hand_value(hand)
        dealer_hand = [deck.pop(), deck.pop()]
        while hand_value(dealer_hand) < 17 and len(deck) > 0:
            dealer_hand.append(deck.pop())
        dealer_val = hand_value(dealer_hand)
        if player_val <= 21 and (dealer_val > 21 or player_val > dealer_val):
            wins += 1
    strategies_compare.append({'strategy': name, 'win_rate': round(wins/n, 4)})

compare_df = pd.DataFrame(strategies_compare)
print("=== Strategy Comparison ===")
print(compare_df.to_string(index=False))

fig2 = px.bar(compare_df, x='strategy', y='win_rate',
              color='win_rate',
              color_continuous_scale='Viridis',
              title='Win Rate by Strategy — BlackJack to Five',
              template='plotly_dark')
fig2.update_layout(width=750, height=430)
fig2.show()

# Export
output_dir = r'C:\Users\Gurveer\ds-portfolio\project-20-blackjack-to-five\outputs'
os.makedirs(output_dir, exist_ok=True)
results_df.to_csv(f'{output_dir}\\simulation_results.csv', index=False)
bust_df.to_csv(f'{output_dir}\\bust_probabilities.csv', index=False)
strategy_df.to_csv(f'{output_dir}\\strategy_optimization.csv', index=False)
compare_df.to_csv(f'{output_dir}\\strategy_comparison.csv', index=False)
print(f"\nAll exports saved to outputs/")

=== Strategy Comparison ===
             strategy  win_rate
Conservative (hit<17)    0.4100
       Basic (hit<18)    0.3990
  Aggressive (hit<19)    0.3570
         Optimal (DP)    0.4014



All exports saved to outputs/


In [7]:
# Cell 7 — Summary
print("""
╔══════════════════════════════════════════════════════════╗
║         BlackJack to Five — Project Summary              ║
╠══════════════════════════════════════════════════════════╣
║  Implementation: Python translation of Java COMP 2000    ║
║  Simulations:    10,000 Monte Carlo games                ║
║  Players:        3 (rotating dealer)                     ║
╠══════════════════════════════════════════════════════════╣
║  Key Findings:                                           ║
║  • Dealer wins 39.5% — confirms house advantage          ║
║  • Avg game length: 9.7 rounds to reach 5 wins           ║
║  • Bust at 20: 100% probability                          ║
║  • Optimal strategy found via dynamic programming        ║
╠══════════════════════════════════════════════════════════╣
║  Stack: Python · NumPy · Pandas · Plotly                 ║
║  Origin: Java COMP 2000 group project (Eclipse)          ║
╚══════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════╗
║         BlackJack to Five — Project Summary              ║
╠══════════════════════════════════════════════════════════╣
║  Implementation: Python translation of Java COMP 2000    ║
║  Simulations:    10,000 Monte Carlo games                ║
║  Players:        3 (rotating dealer)                     ║
╠══════════════════════════════════════════════════════════╣
║  Key Findings:                                           ║
║  • Dealer wins 39.5% — confirms house advantage          ║
║  • Avg game length: 9.7 rounds to reach 5 wins           ║
║  • Bust at 20: 100% probability                          ║
║  • Optimal strategy found via dynamic programming        ║
╠══════════════════════════════════════════════════════════╣
║  Stack: Python · NumPy · Pandas · Plotly                 ║
║  Origin: Java COMP 2000 group project (Eclipse)          ║
╚══════════════════════════════════════════════════════════╝

